In [ ]:
import sys
from pathlib import Path

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/jianghongab/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml

if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    if root_dir.parts[-1:] == ('pollen',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    sys.path.append(root_dir)
print(f"Added the following directory to the PYTHONPATH: {root_dir}")
    
# Set the environment variables from the file <root_dir>/.env
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

Local environment
Added the following directory to the PYTHONPATH: /Users/hongjiang/git/mlfs-book
HopsworksSettings initialized!


In [ ]:
import datetime
import pandas as pd
import xgboost as xgb
import hopsworks
import json
from mlfs.airquality import util
import os
import joblib

In [3]:
project = hopsworks.login(engine="python")
fs = project.get_feature_store()

secrets = hopsworks.get_secrets_api()
location_str = secrets.get_secret("SENSOR_LOCATION_JSON").value
location = json.loads(location_str)
city = location['city']
latitude = location['latitude']
longitude = location['longitude']


2026-01-01 15:01:26,263 INFO: Initializing external client
2026-01-01 15:01:26,263 INFO: Base URL: https://c.app.hopsworks.ai:443
2026-01-01 15:01:26,987 WARNING: UserWarning: The installed hopsworks client version 4.6.0 may not be compatible with the connected Hopsworks backend version 4.2.2. 
To ensure compatibility please install the latest bug fix release matching the minor version of your backend (4.2) by running 'pip install hopsworks==4.2.*'



2026-01-01 15:01:27,810 INFO: Python Engine initialized.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/1292436


In [ ]:
# Configuration: Set number of days to forecast (can be changed to 3, 7, 14, etc.)
# You can also set this in your .env file as FORECAST_DAYS=7
FORECAST_DAYS = int(os.getenv('FORECAST_DAYS', 7))
print(f"Will forecast for the next {FORECAST_DAYS} days")

In [ ]:
today = datetime.datetime.now() - datetime.timedelta(0)
# Generate list of dates for the next N days
forecast_dates = [today + datetime.timedelta(days=i) for i in range(1, FORECAST_DAYS + 1)]
print(f"Forecast dates: {[d.strftime('%Y-%m-%d') for d in forecast_dates]}")

In [ ]:

# 1. Get model
mr = project.get_model_registry()
retrieved_model = mr.get_model(name="grass_pollen_model", version=3)

# 2. Download model
saved_model_dir = retrieved_model.download()
print(f"📦 Model downloaded to: {saved_model_dir}")

# 3. Auto-detect and load model
file_list = os.listdir(saved_model_dir)
print(f"Contains files: {file_list}")

if "pollen_model.pkl" in file_list:
    model_path = os.path.join(saved_model_dir, "pollen_model.pkl")
    model = joblib.load(model_path)
    print("✅ Successfully loaded .pkl model")
elif "model.json" in file_list:
    model_path = os.path.join(saved_model_dir, "model.json")
    model = xgb.XGBRegressor()
    model.load_model(model_path)
    print("✅ Successfully loaded .json model")

Downloading: 0.000%|          | 0/195267 elapsed<00:00 remaining<?

📦 Model downloaded to: /var/folders/vv/h58bs1s95plbyzd7qh_6vd1c0000gn/T/1a657ce2-c214-41d3-a8e7-a51362dc7739/grass_pollen_model/3
Contains files: ['pollen_model.pkl']
✅ Successfully loaded .pkl model


In [ ]:
hourly_df = util.get_hourly_weather_forecast(city, latitude, longitude)
hourly_df = hourly_df.set_index('date')

# We will only make 1 daily prediction, so we will replace the hourly forecasts with a single daily forecast
# We only want the daily weather data, so only get weather at 12:00
daily_df = hourly_df.between_time('11:59', '12:01')
daily_df = daily_df.reset_index()
daily_df['date'] = pd.to_datetime(daily_df['date']).dt.strftime('%Y-%m-%d %H:%M:%S')
daily_df = daily_df.rename(columns={'date': 'datetime_id'}) # Rename column
daily_df['city'] = city
daily_df



Coordinates 59.25°N 18.0°E
Elevation 24.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s


,date,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,city
0,2025-12-25,0.65,0.0,20.176065,285.524170,Stockholm
1,2025-12-26,2.70,0.0,10.188700,302.005341,Stockholm
2,2025-12-27,3.50,0.0,26.208397,307.184784,Stockholm
3,2025-12-28,1.90,0.0,16.279802,305.095886,Stockholm
4,2025-12-29,-1.60,0.0,23.686249,316.847595,Stockholm
5,2025-12-30,-2.50,0.0,20.089161,323.746063,Stockholm
6,2025-12-31,-5.00,0.0,5.116561,309.289368,Stockholm
7,2026-01-01,0.85,1.7,17.102840,149.656830,Stockholm
8,2026-01-02,-2.25,1.5,22.104116,16.073652,Stockholm
9,2026-01-03,-4.00,0.2,22.884789,19.290138,Stockholm


In [ ]:
weather_fg = fs.get_feature_group(
    name='weather',
    version=1,
)

last_forecast_date = forecast_dates[-1]
today_str = today.strftime('%Y-%m-%d %H:%M:%S')
last_forecast_str = last_forecast_date.strftime('%Y-%m-%d %H:%M:%S')

batch_data = weather_fg.filter(weather_fg.datetime_id >= today_str).filter(weather_fg.datetime_id <= last_forecast_str).read()
batch_data


In [ ]:

df_recent = daily_df

# 3. Perform feature engineering locally (no longer dependent on weather_fg.read())
df_recent = df_recent.sort_values('datetime_id')

# Convert datetime_id to datetime for calculations
df_recent['datetime_id'] = pd.to_datetime(df_recent['datetime_id'])

# --- Calculate time features ---
df_recent['day_of_year'] = df_recent['datetime_id'].dt.dayofyear
df_recent['month'] = df_recent['datetime_id'].dt.month
df_recent['is_high_season'] = df_recent['day_of_year'].apply(lambda x: 1 if 140 <= x <= 250 else 0)

# --- Calculate GDD ---
T_base = 5.0
df_recent['gdd_daily'] = df_recent['temperature_2m_mean'].apply(lambda t: max(0, t - T_base))
df_recent['gdd_cumsum'] = df_recent.groupby(df_recent['datetime_id'].dt.year)['gdd_daily'].cumsum()

# --- Calculate key lag features ---
df_recent['precip_lag_1'] = df_recent['precipitation_sum'].shift(1)
df_recent['temp_lag_1'] = df_recent['temperature_2m_mean'].shift(1)
df_recent['wind_lag_1'] = df_recent['wind_speed_10m_max'].shift(1)

# 4. Filter prediction targets: keep only rows from today onwards
batch_data = df_recent[df_recent['datetime_id'] >= datetime.datetime.now().strftime('%Y-%m-%d')].dropna()



In [ ]:
feature_cols = ['temperature_2m_mean', 'precipitation_sum', 'wind_speed_10m_max', 'wind_direction_10m_dominant', 'city', 'day_of_year', 'month', 'is_high_season', 'gdd_daily', 'gdd_cumsum', 'precip_lag_1', 'temp_lag_1', 'wind_lag_1','pollen_lag_1']
batch_data[feature_cols]
batch_data['predicted_pollen'] = model.predict(batch_data[feature_cols].drop(columns=['date_id', 'city'], errors='ignore'))
batch_data

,date,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,city,day_of_year,month,is_high_season,gdd_daily,gdd_cumsum,precip_lag_1,temp_lag_1,wind_lag_1,predicted_pollen
7,2026-01-01,0.85,1.7,17.102840,149.656830,Stockholm,1,1,0,0,0,0.0,-5.00,5.116561,0.000899
8,2026-01-02,-2.25,1.5,22.104116,16.073652,Stockholm,2,1,0,0,0,1.7,0.85,17.102840,0.011892
9,2026-01-03,-4.00,0.2,22.884789,19.290138,Stockholm,3,1,0,0,0,1.5,-2.25,22.104116,0.020034
10,2026-01-04,-5.10,0.0,11.159999,360.000000,Stockholm,4,1,0,0,0,0.2,-4.00,22.884789,0.006500
11,2026-01-05,-8.10,0.0,13.276144,319.398773,Stockholm,5,1,0,0,0,0.0,-5.10,11.159999,0.008704
12,2026-01-06,-5.45,0.0,8.942214,220.100845,Stockholm,6,1,0,0,0,0.0,-8.10,13.276144,0.006011
13,2026-01-07,-3.85,0.4,22.104116,210.323517,Stockholm,7,1,0,0,0,0.0,-5.45,8.942214,0.040152


In [ ]:
batch_data['city'] = city
batch_data['days_before_forecast_day'] = range(1, len(batch_data)+1)
batch_data = batch_data.sort_values(by=['datetime_id'])
batch_data

,date,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,city,day_of_year,month,is_high_season,gdd_daily,gdd_cumsum,precip_lag_1,temp_lag_1,wind_lag_1,predicted_pollen,days_before_forecast_day
7,2026-01-01,0.85,1.7,17.102840,149.656830,Stockholm,1,1,0,0,0,0.0,-5.00,5.116561,0.000899,1
8,2026-01-02,-2.25,1.5,22.104116,16.073652,Stockholm,2,1,0,0,0,1.7,0.85,17.102840,0.011892,2
9,2026-01-03,-4.00,0.2,22.884789,19.290138,Stockholm,3,1,0,0,0,1.5,-2.25,22.104116,0.020034,3
10,2026-01-04,-5.10,0.0,11.159999,360.000000,Stockholm,4,1,0,0,0,0.2,-4.00,22.884789,0.006500,4
11,2026-01-05,-8.10,0.0,13.276144,319.398773,Stockholm,5,1,0,0,0,0.0,-5.10,11.159999,0.008704,5
12,2026-01-06,-5.45,0.0,8.942214,220.100845,Stockholm,6,1,0,0,0,0.0,-8.10,13.276144,0.006011,6
13,2026-01-07,-3.85,0.4,22.104116,210.323517,Stockholm,7,1,0,0,0,0.0,-5.45,8.942214,0.040152,7


In [ ]:
monitor_fg = fs.get_or_create_feature_group(
    name='grass_pollen_predictions',
    description='Grass pollen prediction monitoring',
    version=1,
    primary_key=['city','datetime_id','days_before_forecast_day'],
    event_time="date"
)

In [10]:
monitor_fg.insert(batch_data, wait=True)

Uploading Dataframe: 100.00% |█| Rows 7/7 | Elapsed Time: 00:01 | Remaining Time


Launching job: pollen_predictions_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1292436/jobs/named/pollen_predictions_1_offline_fg_materialization/executions
2026-01-01 15:03:26,762 INFO: Waiting for execution to finish. Current state: INITIALIZING. Final status: UNDEFINED
2026-01-01 15:03:29,959 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-01-01 15:05:05,421 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2026-01-01 15:05:05,582 INFO: Waiting for log aggregation to finish.
2026-01-01 15:05:14,183 INFO: Execution finished successfully.


(Job('pollen_predictions_1_offline_fg_materialization', 'SPARK'), None)